In [1]:
import sys
import time
from statistics import median
from timeit import repeat

In [2]:
APPEND_COUNT = 20_000
CHUNK = 5_000
TABLE_CAPACITY = 10_000
TABLE_ENTRIES = 50
COMPACT_CAPACITY = 128
MEMBERSHIP_SIZES = (300_000, 1_000_000)
LIST_REPEATS = 3
SET_REPEATS = 1000
BENCHMARK_REPEATS = 5

# ДЗ №2. Структуры данных: массив, список, хеш

Курс «Алгоритмы и структуры данных» · по материалам лекции 2

**Фамилия, группа:** Ибрагимов, МПМ-26-1

## Как работать с этим ноутбуком

- Демо-ячейки уже исполнены — выводы видны сразу. Прогоните их сами: Shift+Enter, сверху вниз.
- Ячейки **«### Ответ»** — ваши: пишите в них, заглушку затрите.
- Пометка `# <-- можно менять здесь` — приглашение поиграть с числами.
- Нужен только Python 3.9+, без пакетов и интернета.
- Точные числа у вас будут другими — важен порядок величин, не миллисекунды.

## Шпаргалка с лекции

| Структура | Сильная сторона | Слабая сторона |
|---|---|---|
| Массив | чтение по индексу O(1) | вставка/удаление O(n) |
| Связанный список | вставка/удаление O(1) | чтение O(n) |
| Стек / очередь | простой порядок обработки | доступ только к концам |
| Хеш-таблица | поиск/вставка/удаление O(1) в среднем | нет порядка, худший случай O(n) |

**Амортизированная стоимость** — средняя стоимость одной операции, когда
редкие дорогие операции размазываются на много дешёвых. Динамический массив
(Python `list`) удваивается при переполнении: вставка в конец — O(1) в среднем.

---

## Задача 1. Журнал транзакций

Финансовое приложение: тысячи транзакций в день **дописываются в конец**
журнала, чтение записей редкое. Что взять — массив или связанный список?

Сначала рассуждение, потом эксперимент: ячейка ниже сравнивает вставку
в конец и в начало растущего списка — порциями, чтобы было видно,
как ведёт себя время каждой порции.

In [3]:
def time_chunks(insert_at_end):
    data = []
    times = []
    for _ in range(APPEND_COUNT // CHUNK):
        t0 = time.perf_counter()
        for i in range(CHUNK):
            if insert_at_end:
                data.append(i)  # в конец
            else:
                data.insert(0, i)  # в начало
        times.append(time.perf_counter() - t0)
    return times


tail = time_chunks(True)
head = time_chunks(False)

for k, (a, b) in enumerate(zip(tail, head), 1):
    print(f"порция {k}: в конец {a * 1000:6.1f} мс, в начало {b * 1000:7.1f} мс")

порция 1: в конец    0.4 мс, в начало     5.4 мс
порция 2: в конец    0.4 мс, в начало    11.6 мс
порция 3: в конец    0.5 мс, в начало    15.0 мс
порция 4: в конец    0.4 мс, в начало    19.9 мс


### Ответ (задача 1)

Для указанной нагрузки выбираю динамический массив: добавление в конец имеет амортизированную стоимость O(1), хотя еще лучше было бы использовать deque.
Связанный список с указателем на хвост тоже допустим:
он добавляет узел за O(1), но требует памяти для ссылок и обычно отдельных аллокаций узлов.
Сам факт большого числа вставок в конец ещё не доказывает преимущество связанного списка.

Готовый эксперимент сравнивает две операции Python list, а не list со связанным списком.
При вставке в начало приходится сдвигать ссылки: одна операция занимает O(n), поэтому поздние порции дороже.
При добавлении в конец средняя стоимость операции остаётся O(1), хотя отдельное расширение может быть дорогим.

**Практическая справка.** Для журнала на диске можно рассмотреть LSM-дерево или B-tree, если нужны
индексированные запросы и соответствующие гарантии хранения. Для простого дописывания достаточно последовательного
журнала; выбор дерева требует дополнительных условий о чтении, обновлениях и долговечности.
В памяти deque в CPython использует связанные блоки по 64 указателя: это уменьшает число аллокаций по сравнению
с отдельным узлом на запись и не требует перемещения всего массива ссылок при росте.
Такой вариант полезен, если важны задержки отдельных добавлений или операции с обоих концов,
но преимущество над list для этой нагрузки нужно измерять.

---

## Задача 2. Топ-10 злодеев

Сайт ведёт рейтинг из 1 000 000 записей. Пользователь листает кнопкой Next —
читает записи **подряд, начиная с первой**. Массив или связанный список?

Маленький эксперимент про чтение «подряд с начала»: сколько стоит
прочитать первые 10 записей из миллиона.

In [4]:
million = list(range(1_000_000))

t0 = time.perf_counter()
top10 = million[:10]  # первые 10, подряд с начала
t_first10 = time.perf_counter() - t0

t0 = time.perf_counter()
everything = million[:]  # а так — пройти весь миллион
t_all = time.perf_counter() - t0

print(f"прочитать первые 10:    {t_first10 * 1_000_000:8.1f} мкс")
print(f"прочитать весь миллион: {t_all * 1000:8.1f} мс")
print(f"разница: {t_all / t_first10:,.0f} раз")

прочитать первые 10:        61.7 мкс
прочитать весь миллион:      6.9 мс
разница: 111 раз


### Ответ (задача 2)

Выбираю массив: для показа очередных десяти записей читаются десять соседних позиций без переходов
по цепочке узлов. Требуется последовательный доступ, однако массив также позволяет сразу вычислить начало страницы.
Связанный список допустим, но чтение последовательно Х элементов из списка дороже, чем Х элементов из массива. 

Компактное размещение уменьшает зависимые переходы по указателям и помогает кешу процессора.

В этом запуске первые десять ссылок скопированы за 61.75 мкс,
весь миллион — за 6.870 мс, примерно в 111 раз дольше.

---

## Задача 3. Хеш-таблица, заполненная на 5%

Хеш-таблица на 10 000 ячеек, в ней 50 записей — заполнение 0.05.
Что с ней не так и что делать?

Это задача-рассуждение, но потыкать всё равно интересно: ниже — игрушечная
хеш-таблица с открытой адресацией, как на лекции.

In [5]:
class SimpleHashTable:
    """Игрушечная хеш-таблица с открытой адресацией — как на лекции."""

    def __init__(self, capacity):
        self.capacity = capacity
        self.cells = [None] * capacity
        self.size = 0

    def put(self, key):
        i = hash(key) % self.capacity
        while self.cells[i] is not None:
            i = (i + 1) % self.capacity  # коллизия — шаг вперёд
        self.cells[i] = key
        self.size += 1

    def load_factor(self):
        return self.size / self.capacity


t = SimpleHashTable(TABLE_CAPACITY)
for k in range(TABLE_ENTRIES):
    t.put(f"запись-{k}")

print(f"записей: {t.size}, ячеек: {t.capacity:,}")
print(f"заполнение: {t.load_factor():.3f}")
print(f"память под ячейки: {sys.getsizeof(t.cells) / 1024:.0f} КБ")
print(f"полезно занято: {t.size * sys.getsizeof(None) / 1024:.1f} КБ")

compact_table = SimpleHashTable(COMPACT_CAPACITY)
for stored_key in t.cells:
    if stored_key is not None:
        compact_table.put(stored_key)
original_keys = {value for value in t.cells if value is not None}
compact_keys = {value for value in compact_table.cells if value is not None}
assert original_keys == compact_keys and compact_table.size == t.size
for stored_key in original_keys:
    position = hash(stored_key) % compact_table.capacity
    for attempt in range(compact_table.capacity):
        if compact_table.cells[position] is None:
            raise AssertionError("Ключ потерян после перехеширования")
        if compact_table.cells[position] == stored_key:
            break
        position = (position + 1) % compact_table.capacity
    else:
        raise AssertionError("Ключ не найден")
print("После переноса: ячеек", compact_table.capacity, "заполнение", compact_table.load_factor())
print("Массив ссылок до/после, байт:", sys.getsizeof(t.cells), sys.getsizeof(compact_table.cells))

записей: 50, ячеек: 10,000
заполнение: 0.005
память под ячейки: 78 КБ
полезно занято: 0.8 КБ
После переноса: ячеек 128 заполнение 0.390625
Массив ссылок до/после, байт: 80056 1080


### Ответ (задача 3)

Само по себе низкое заполнение - не ошибка: память могли заранее выделить под ожидаемый рост.
Если роста не будет и память важна, можно пересоздать словарь. 
Если текущий расход памяти несущественен, пересоздание может вообще не понадобиться.

При уменьшении ёмкости меняется hash(key) % capacity, поэтому нельзя просто обрезать массив ячеек.
Ниже новая таблица заполняется через put. Перенос занимает как минимум O(n) работы; стоимость поиска
при этом зависит от коллизий, поэтому размер выбирается с учётом реализации и последующей нагрузки.
Выражение sys.getsizeof(None) в исходном примере не измеряет полезную память записей: ячейки содержат ссылки
на строки. Для сравнения ниже измеряются именно массивы ссылок одинаковым способом, без общей памяти строк.

После переноса все 50 ключей сохранены и находятся поиском с открытой адресацией.
Размер массива ячеек уменьшился с 80056 до 1080 байт,
примерно в 74.1 раза; строки и остальные поля таблицы сюда не входят.

---

## Задача 4 (со звёздочкой). Голосовал ли избиратель?

По списку из 1 000 000 фамилий нужно много раз проверять: голосовал ли
избиратель с такой-то фамилией? Какая структура данных подходит
и какой выигрыш против линейного поиска?

Сначала — гипотеза письменно. Потом замер: для скорости возьмём 300 000
фамилий (полный миллион тоже ок, просто ячейка будет работать дольше —
попробуйте оба варианта).

Гипотеза до замера: множество будет быстрее линейного поиска;
при переходе от 300 000 к миллиону записей время линейного поиска вырастет примерно в 10/3 раза.

In [6]:
def measure_membership(size: int) -> dict[str, float]:
    surnames = [f"фамилия-{index}" for index in range(size)]
    voters_list = surnames[:]
    voters_set = set(surnames)
    probe = surnames[-1]
    assert probe in voters_list and probe in voters_set
    assert "отсутствующая-запись" not in voters_set
    linear = median(repeat(lambda: probe in voters_list, number=LIST_REPEATS, repeat=BENCHMARK_REPEATS)) / LIST_REPEATS
    hashed = median(repeat(lambda: probe in voters_set, number=SET_REPEATS, repeat=BENCHMARK_REPEATS)) / SET_REPEATS
    return {"size": size, "linear": linear, "hashed": hashed, "speedup": linear / hashed}


membership_results = [measure_membership(size) for size in MEMBERSHIP_SIZES]
for row in membership_results:
    print(
        f"n={row['size']:,}: список {row['linear'] * 1000:.3f} мс; "
        f"множество {row['hashed'] * 1e6:.3f} мкс; выигрыш {row['speedup']:.0f} раз"
    )
predicted_million = membership_results[0]["linear"] * MEMBERSHIP_SIZES[1] / MEMBERSHIP_SIZES[0]
print(f"Экстраполяция линейного поиска на миллион: {predicted_million * 1000:.3f} мс")

n=300,000: список 5.864 мс; множество 0.049 мкс; выигрыш 119576 раз
n=1,000,000: список 20.640 мс; множество 0.049 мкс; выигрыш 417303 раз
Экстраполяция линейного поиска на миллион: 19.546 мс


### Ответ (задача 4)

Для многократной проверки наличия выбираю хеш-множество set: значения не нужны, достаточно хранить ключи.
Построение требует O(n) времени и дополнительной памяти, поиск — ожидаемо O(1)
при хорошем распределении хешей.

Фильтр Блума перед точным множеством или мапой может отсеивать отрицательные запросы,
если точная проверка дорогая и промахов много. Положительный ответ фильтра нужно проверять в точной структуре:
возможны ложноположительные срабатывания. Для стандартного фильтра, корректно пополняемого вставками,
отрицательный ответ исключает наличие ключа. Один фильтр не подходит для точного учёта голосования.
Для множества, целиком находящегося в памяти, дополнительное хеширование может не окупиться.

В замере структура построена заранее: измеряется поиск последнего элемента, а не стоимость построения.
Использована медиана пяти повторов с несколькими поисками внутри каждого; числа зависят от среды и кешей.
Экстраполяция O(n) относится к списку, а не к гарантированному коэффициенту ускорения множества.

Пример применения фильтров: [RocksDB](https://github.com/facebook/rocksdb/wiki/RocksDB-Bloom-Filter).

Результаты сохранённого запуска:

| Записей | Линейный поиск, мс | Поиск в set, мкс | Отношение времени |
| --- | --- | --- | --- |
| 300000 | 5.864 | 0.049 | 119576 |
| 1000000 | 20.640 | 0.049 | 417303 |

Линейная экстраполяция первого замера дала 19.546 мс на миллион;
прямой замер дал 20.640 мс. Порядок величин согласуется с линейным ростом.
В эксперименте повторяется один и тот же существующий строковый объект: его хеш уже вычислен,
а рабочие данные прогреты повторениями. Поэтому большой коэффициент ускорения относится именно
к этому стенду; для новых строк, смеси попаданий и промахов, холодных кешей и построения множества
нужны отдельные измерения. Порядок роста не выводится из одного коэффициента.

---

## Сдача

Чек-лист перед отправкой:

- имя и группа вписаны в титул;
- демо-ячейки прогнаны, выводы на месте;
- все ячейки «### Ответ» заполнены;
- файл сохранён и отправлен преподавателю (.ipynb).

Если какая-то ячейка не запускается или что-то пошло не так — напишите
прямо в этой ячейке, что произошло: это нормально.